# Phase 2 Notebook 3: Multi-Horizon Signal Scoring

This notebook loads structurally approved candidate signals from SQLite, computes forward returns from clean close prices, scores each signal across multiple horizons, applies a preliminary scoring gate, and writes the scoring artifacts back to SQLite.

Scope boundary: this notebook does not run walk-forward validation, stress testing, or final survivor selection.

## Imports and Config

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import gc
import sys
import time

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == '2-Phase 2_Signal Expansion':
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db import get_db_path, load_price_table
from src.forward_returns import make_forward_returns, validate_forward_return_panels
from src.scoring_storage import SCORING_TABLES, save_scoring_outputs
from src.signal_scoring import (
    apply_preliminary_scoring_gate,
    build_best_horizon_summary,
    build_scoring_family_summary,
    build_signal_score_summary,
    score_signal_library_multi_horizon,
)
from src.signal_storage import (
    load_candidate_signal_quality_gate,
    load_and_pivot_signal_panels_by_names,
    load_candidate_signals_by_names,
)

HORIZONS = [1, 5, 10, 20]
SCORING_VERSION = 'phase2_signal_scoring_v2'
IC_METHOD = 'spearman'
MIN_SCORE_OBS = 1000
MIN_GATE_OBS = 10000

sqlite_db_path = get_db_path()
print(f'SQLite database: {sqlite_db_path}')


SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db


## Create Run ID

In [2]:
run_timestamp = datetime.now(timezone.utc).replace(microsecond=0)
run_id = f"signal_scoring_{run_timestamp.strftime('%Y%m%d_%H%M%S')}"

print(f'run_id: {run_id}')
print(f'run_timestamp_utc: {run_timestamp.isoformat()}')
print(f'scoring_version: {SCORING_VERSION}')

run_id: signal_scoring_20260510_222922
run_timestamp_utc: 2026-05-10T22:29:22+00:00
scoring_version: phase2_signal_scoring_v2


## Load Approved Candidates

In [3]:
quality_gate = load_candidate_signal_quality_gate(current=True, db_path=sqlite_db_path)
approved_candidates = quality_gate.loc[
    quality_gate['status'].eq('APPROVED_FOR_SCORING')
].copy()

if approved_candidates.empty:
    raise ValueError('No APPROVED_FOR_SCORING rows found in candidate_signal_quality_gate_current.')

approved_signal_names = approved_candidates['signal_name'].dropna().sort_values().tolist()
print(f'Approved candidate signals: {len(approved_signal_names)}')
display(approved_candidates[['signal_name', 'signal_family', 'n_dates', 'n_tickers', 'missing_pct', 'finite_pct', 'status']])

Approved candidate signals: 23


,signal_name,signal_family,n_dates,n_tickers,missing_pct,finite_pct,status
85,relative_return_rank_20,cross_sectional_relative_value,2098,478,0.123550,0.876450,APPROVED_FOR_SCORING
86,relative_return_zscore_60,cross_sectional_relative_value,2098,478,0.323913,0.676087,APPROVED_FOR_SCORING
87,residual_return_vs_universe_20,cross_sectional_relative_value,2098,478,0.361392,0.638608,APPROVED_FOR_SCORING
88,overnight_gap_reversal_1,true_short_term_reversal,2098,478,0.110643,0.889357,APPROVED_FOR_SCORING
89,intraday_reversal_strength_1,true_short_term_reversal,2098,478,0.107308,0.892692,APPROVED_FOR_SCORING
90,three_day_overextension_reversal,true_short_term_reversal,2098,478,0.111843,0.888157,APPROVED_FOR_SCORING
91,vol_surprise_20_60,volatility_structure,2098,478,0.128632,0.871368,APPROVED_FOR_SCORING
92,vol_of_vol_20,volatility_structure,2098,478,0.119418,0.880582,APPROVED_FOR_SCORING
93,range_expansion_failure_5,volatility_structure,2098,478,0.127500,0.872500,APPROVED_FOR_SCORING
94,dollar_volume_shock_20,liquidity_flow,2098,478,0.173367,0.826633,APPROVED_FOR_SCORING


## Load Candidate Signals

In [4]:
smoke_signal_name = approved_signal_names[0]
smoke_start = time.perf_counter()
smoke_signal_long = load_candidate_signals_by_names(
    [smoke_signal_name],
    current=True,
    db_path=sqlite_db_path,
    chunksize=500_000,
)
print(
    f'Smoke loaded 1 approved signal ({smoke_signal_name}): '
    f'{len(smoke_signal_long):,} rows in {time.perf_counter() - smoke_start:.3f}s'
)
del smoke_signal_long
gc.collect()

signal_panels = load_and_pivot_signal_panels_by_names(
    approved_signal_names,
    current=True,
    db_path=sqlite_db_path,
    duplicate_policy='raise',
    chunksize=500_000,
)

if len(signal_panels) != len(approved_signal_names):
    raise ValueError(
        f'Expected {len(approved_signal_names)} signal panels, got {len(signal_panels)}.'
    )

panel_validation_failures = []
for signal_name, panel in signal_panels.items():
    if panel.empty:
        panel_validation_failures.append(f'{signal_name}: empty panel')
    if not panel.index.is_unique:
        panel_validation_failures.append(f'{signal_name}: non-unique Date index')
    if panel.shape[1] == 0:
        panel_validation_failures.append(f'{signal_name}: no ticker columns')

if panel_validation_failures:
    raise ValueError('Signal panel validation failed: ' + '; '.join(panel_validation_failures[:10]))

print(f'Built and validated signal panels: {len(signal_panels):,}')

load_candidate_signals_by_names: table=candidate_signals_current, requested_signal_names=1, rows_returned=1,002,844, elapsed_seconds=1.499, memory_before_mb=148.0, memory_after_mb=737.8
Smoke loaded 1 approved signal (close_position_reversal_5): 1,002,844 rows in 1.517s
load_candidate_signals_by_names: table=candidate_signals_current, requested_signal_names=1, rows_returned=1,002,844, elapsed_seconds=1.252, memory_before_mb=351.8, memory_after_mb=813.0
approved_signal_long[close_position_reversal_5] duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
approved_signal_long duplicate diagnostics for ['signal_name', 'Date', 'ticker']: duplicate_rows=0, duplicate_groups=0
load_and_pivot_signal_panels_by_names: signal=1/23 close_position_reversal_5, load_seconds=1.271, pivot_seconds=0.467, panel_shape=(2098, 478), memory_after_mb=599.8
load_candidate_signals_by_names: table=candidate_signals_current, requested_signal_names=1, rows_returned=1,002,

## Pivot Approved Signals to Panels

In [5]:
panel_shapes = pd.DataFrame(
    [
        {'signal_name': name, 'n_dates': panel.shape[0], 'n_tickers': panel.shape[1]}
        for name, panel in signal_panels.items()
    ]
)
display(panel_shapes)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after pivoting approved signals')

,signal_name,n_dates,n_tickers
0,close_position_reversal_5,2098,478
1,dollar_volume_shock_20,2098,478
2,expanded_beta_adjusted_residual_20,2098,478
3,expanded_distance_ma_10,2098,478
4,expanded_distance_ma_20,2098,478
5,expanded_residual_market_return_20,2098,478
6,expanded_reversal_1d,2098,478
7,expanded_reversal_3d,2098,478
8,expanded_reversal_5d,2098,478
9,expanded_zscore_reversal_20,2098,478


## Load Clean Close Prices

In [6]:
close_prices = load_price_table('clean_close_prices_current', db_path=sqlite_db_path)

print(f'Close price panel shape: {close_prices.shape}')
print(f'Date range: {close_prices.index.min()} to {close_prices.index.max()}')
display(close_prices.tail())

Close price panel shape: (2098, 478)
Date range: 2018-01-02 00:00:00 to 2026-05-07 00:00:00


,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2026-05-01,NaN,280.140015,206.600006,141.660004,89.459999,NaN,179.830002,250.710007,397.690002,NaN,...,NaN,NaN,NaN,82.580002,152.750000,115.370003,NaN,NaN,NaN,114.160004
2026-05-04,NaN,276.829987,208.160004,138.860001,87.540001,NaN,180.119995,253.960007,397.019989,NaN,...,NaN,NaN,NaN,81.169998,153.690002,114.839996,NaN,NaN,NaN,112.680000
2026-05-05,NaN,284.179993,206.110001,139.729996,87.169998,NaN,179.009995,255.619995,404.769989,NaN,...,NaN,NaN,NaN,81.449997,154.880005,116.389999,NaN,NaN,NaN,112.540001
2026-05-06,NaN,287.510010,205.029999,139.880005,86.300003,NaN,174.570007,250.169998,415.630005,NaN,...,NaN,NaN,NaN,80.550003,148.690002,118.589996,NaN,NaN,NaN,111.220001
2026-05-07,NaN,287.440002,202.710007,140.460007,87.010002,NaN,180.190002,256.510010,408.519989,NaN,...,NaN,NaN,NaN,80.430000,146.580002,115.639999,NaN,NaN,NaN,87.309998


## Compute Forward Returns

In [7]:
forward_returns = make_forward_returns(close_prices, HORIZONS)
validate_forward_return_panels(forward_returns, close_prices)

forward_return_shapes = pd.DataFrame(
    [
        {'horizon': horizon, 'n_dates': panel.shape[0], 'n_tickers': panel.shape[1]}
        for horizon, panel in forward_returns.items()
    ]
)
display(forward_return_shapes)

,horizon,n_dates,n_tickers
0,1,2098,478
1,5,2098,478
2,10,2098,478
3,20,2098,478


## Score Approved Signals Across Horizons

In [8]:
scores = score_signal_library_multi_horizon(
    signals=signal_panels,
    forward_returns=forward_returns,
    metadata=approved_candidates[['signal_name', 'signal_family', 'signal_version']],
    horizons=HORIZONS,
    method=IC_METHOD,
    min_obs=MIN_SCORE_OBS,
)

scores = scores.sort_values(['signal_name', 'horizon']).reset_index(drop=True)
display(scores)

,signal_name,horizon,method,n_obs,mean_ic,median_ic,ic_std,ic_ir,hit_rate,positive_ic_rate,missing_pct,signal_family,signal_version
0,close_position_reversal_5,1,spearman,600770,0.006244,0.000382,0.190790,0.032728,0.499689,0.500491,0.400934,microstructure_lite,phase2_orthogonal_signals_v2
1,close_position_reversal_5,5,spearman,589206,0.007745,0.003505,0.179986,0.043031,0.496022,0.508358,0.412465,microstructure_lite,phase2_orthogonal_signals_v2
2,close_position_reversal_5,10,spearman,578401,0.006744,0.008758,0.173888,0.038786,0.494546,0.517003,0.423239,microstructure_lite,phase2_orthogonal_signals_v2
3,close_position_reversal_5,20,spearman,562984,0.008850,0.009159,0.166689,0.053094,0.494270,0.526994,0.438613,microstructure_lite,phase2_orthogonal_signals_v2
4,dollar_volume_shock_20,1,spearman,594678,0.002990,0.004823,0.092843,0.032205,0.500327,0.523360,0.407008,liquidity_flow,phase2_orthogonal_signals_v2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,vol_of_vol_20,20,spearman,563319,0.015976,0.016122,0.124049,0.128788,0.501826,0.553403,0.438279,volatility_structure,phase2_orthogonal_signals_v2
88,vol_surprise_20_60,1,spearman,597068,0.000074,-0.000443,0.138973,0.000530,0.495040,0.498509,0.404625,volatility_structure,phase2_orthogonal_signals_v2
89,vol_surprise_20_60,5,spearman,584855,0.003646,0.002618,0.137257,0.026566,0.492747,0.506972,0.416804,volatility_structure,phase2_orthogonal_signals_v2
90,vol_surprise_20_60,10,spearman,573488,0.006110,0.011187,0.129978,0.047010,0.490957,0.536695,0.428138,volatility_structure,phase2_orthogonal_signals_v2


## Apply Preliminary Scoring Gate

In [9]:
scoring_gate = apply_preliminary_scoring_gate(
    scores,
    min_abs_mean_ic=0.025,
    min_abs_ic_ir=0.10,
    positive_ic_rate_upper=0.53,
    positive_ic_rate_lower=0.47,
    watchlist_abs_mean_ic=0.012,
    min_n_obs=MIN_GATE_OBS,
)
score_summary = build_signal_score_summary(scores)
best_horizon_summary = build_best_horizon_summary(scores)
family_summary = build_scoring_family_summary(scores)

status_counts = scoring_gate['status'].value_counts().rename_axis('status').reset_index(name='count')

display(status_counts)
display(scoring_gate.sort_values('abs_mean_ic', ascending=False))
display(score_summary)
display(best_horizon_summary)
display(family_summary)


,status,count
0,REJECTED_LOW_SIGNAL,76
1,WATCHLIST,16


,signal_name,horizon,method,n_obs,mean_ic,median_ic,ic_std,ic_ir,hit_rate,positive_ic_rate,missing_pct,signal_family,signal_version,abs_mean_ic,abs_ic_ir,signal_direction,signal_strength,status,scoring_gate_notes
24,expanded_reversal_1d,1,spearman,617177,0.017276,0.013111,0.204011,0.084684,0.505222,0.529553,0.384573,mean_reversion,phase2_expanded_discovery_v1,0.017276,0.084684,POSITIVE_EDGE,WEAK,WATCHLIST,Meets watchlist absolute IC and observation th...
23,expanded_residual_market_return_20,20,spearman,551481,0.016748,0.011219,0.188325,0.088930,0.501801,0.528145,0.450083,residual_relative_value,phase2_expanded_discovery_v1,0.016748,0.088930,POSITIVE_EDGE,WEAK,WATCHLIST,Meets watchlist absolute IC and observation th...
87,vol_of_vol_20,20,spearman,563319,0.015976,0.016122,0.124049,0.128788,0.501826,0.553403,0.438279,volatility_structure,phase2_orthogonal_signals_v2,0.015976,0.128788,POSITIVE_EDGE,WEAK,WATCHLIST,Meets watchlist absolute IC and observation th...
28,expanded_reversal_3d,1,spearman,610402,0.014939,0.010738,0.212333,0.070354,0.503985,0.521886,0.391329,mean_reversion,phase2_expanded_discovery_v1,0.014939,0.070354,POSITIVE_EDGE,WEAK,WATCHLIST,Meets watchlist absolute IC and observation th...
67,range_expansion_failure_5,20,spearman,569312,0.014333,0.012657,0.127624,0.112310,0.497760,0.543756,0.432303,volatility_structure,phase2_orthogonal_signals_v2,0.014333,0.112310,POSITIVE_EDGE,WEAK,WATCHLIST,Meets watchlist absolute IC and observation th...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,vol_of_vol_20,1,spearman,603611,0.000629,-0.001114,0.128798,0.004881,0.498515,0.495079,0.398101,volatility_structure,phase2_orthogonal_signals_v2,0.000629,0.004881,POSITIVE_EDGE,NO_SIGNAL,REJECTED_LOW_SIGNAL,Fails preliminary predictive scoring thresholds.
56,price_impact_proxy_20,1,spearman,595338,-0.000585,-0.000988,0.136607,-0.004281,0.496717,0.497018,0.406350,liquidity_flow,phase2_orthogonal_signals_v2,0.000585,0.004281,NEGATIVE_EDGE_REVERSE_SIGNAL,NO_SIGNAL,REJECTED_LOW_SIGNAL,Fails preliminary predictive scoring thresholds.
5,dollar_volume_shock_20,5,spearman,582882,0.000087,-0.000060,0.088866,0.000974,0.497501,0.499502,0.418771,liquidity_flow,phase2_orthogonal_signals_v2,0.000087,0.000974,POSITIVE_EDGE,NO_SIGNAL,REJECTED_LOW_SIGNAL,Fails preliminary predictive scoring thresholds.
88,vol_surprise_20_60,1,spearman,597068,0.000074,-0.000443,0.138973,0.000530,0.495040,0.498509,0.404625,volatility_structure,phase2_orthogonal_signals_v2,0.000074,0.000530,POSITIVE_EDGE,NO_SIGNAL,REJECTED_LOW_SIGNAL,Fails preliminary predictive scoring thresholds.


,signal_name,n_horizons_scored,best_horizon,best_abs_mean_ic,best_mean_ic,mean_abs_mean_ic,avg_positive_ic_rate,avg_hit_rate,max_n_obs,avg_missing_pct
0,expanded_reversal_1d,4,1,0.017276,0.017276,0.010394,0.515895,0.501654,617177,0.404112
1,expanded_residual_market_return_20,4,20,0.016748,0.016748,0.011072,0.509412,0.500487,575856,0.437343
2,vol_of_vol_20,4,20,0.015976,0.015976,0.009136,0.521940,0.500195,603611,0.417175
3,expanded_reversal_3d,4,1,0.014939,0.014939,0.010668,0.517506,0.500736,610402,0.409760
4,range_expansion_failure_5,4,20,0.014333,0.014333,0.010371,0.534136,0.497669,609572,0.411225
5,expanded_distance_ma_10,4,1,0.014215,0.014215,0.011683,0.520525,0.499778,609574,0.410432
6,range_compression_breakout_10,4,20,0.014039,-0.014039,0.007730,0.481849,0.496837,600684,0.418888
7,intraday_reversal_strength_1,4,1,0.014005,0.014005,0.008474,0.517394,0.501475,613469,0.407702
8,failed_breakout_reversal_20,4,20,0.013674,-0.013674,0.011039,0.477343,0.500166,609573,0.411225
9,residual_return_vs_universe_20,4,20,0.013551,-0.013551,0.011938,0.478178,0.497504,592265,0.425907


,signal_name,signal_family,best_horizon,best_mean_ic,best_abs_mean_ic,best_ic_ir,best_positive_ic_rate,best_hit_rate,signal_direction,signal_strength
0,expanded_reversal_1d,mean_reversion,1,0.017276,0.017276,0.084684,0.529553,0.505222,POSITIVE_EDGE,WEAK
1,expanded_residual_market_return_20,residual_relative_value,20,0.016748,0.016748,0.088930,0.528145,0.501801,POSITIVE_EDGE,WEAK
2,vol_of_vol_20,volatility_structure,20,0.015976,0.015976,0.128788,0.553403,0.501826,POSITIVE_EDGE,WEAK
3,expanded_reversal_3d,mean_reversion,1,0.014939,0.014939,0.070354,0.521886,0.503985,POSITIVE_EDGE,WEAK
4,range_expansion_failure_5,volatility_structure,20,0.014333,0.014333,0.112310,0.543756,0.497760,POSITIVE_EDGE,WEAK
5,expanded_distance_ma_10,mean_reversion,1,0.014215,0.014215,0.065670,0.522137,0.503944,POSITIVE_EDGE,WEAK
6,range_compression_breakout_10,microstructure_lite,20,-0.014039,0.014039,-0.096098,0.464586,0.496183,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK
7,intraday_reversal_strength_1,true_short_term_reversal,1,0.014005,0.014005,0.080810,0.530553,0.504589,POSITIVE_EDGE,WEAK
8,failed_breakout_reversal_20,microstructure_lite,20,-0.013674,0.013674,-0.097765,0.456244,0.499829,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK
9,residual_return_vs_universe_20,cross_sectional_relative_value,20,-0.013551,0.013551,-0.077233,0.481038,0.497279,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK


,signal_family,n_signal_horizons,avg_abs_mean_ic,max_abs_mean_ic,best_signal_name,best_horizon
0,mean_reversion,24,0.010422,0.017276,expanded_reversal_1d,1
1,residual_relative_value,8,0.008015,0.016748,expanded_residual_market_return_20,20
2,volatility_structure,12,0.007964,0.015976,vol_of_vol_20,20
3,microstructure_lite,12,0.008722,0.014039,range_compression_breakout_10,20
4,true_short_term_reversal,12,0.005701,0.014005,intraday_reversal_strength_1,1
5,cross_sectional_relative_value,12,0.008923,0.013551,residual_return_vs_universe_20,20
6,liquidity_flow,12,0.004905,0.011528,price_impact_proxy_20,20


## Save Scoring Outputs to SQLite

In [10]:
saved_paths = save_scoring_outputs(
    scores=scores,
    summary=score_summary,
    gate=scoring_gate,
    best_horizon=best_horizon_summary,
    family_summary=family_summary,
    db_path=sqlite_db_path,
    run_id=run_id,
    scoring_version=SCORING_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            'artifact': artifact,
            'current_table': tables[0],
            'history_table': tables[1],
            'sqlite_path': str(saved_paths[artifact]),
        }
        for artifact, tables in SCORING_TABLES.items()
    ]
)

display(sqlite_tables_written)


,artifact,current_table,history_table,sqlite_path
0,scores,signal_scores_current,signal_scores_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,signal_score_summary_current,signal_score_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,gate,signal_scoring_gate_current,signal_scoring_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,best_horizon,signal_best_horizon_current,signal_best_horizon_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,family_summary,signal_scoring_family_summary_current,signal_scoring_family_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## Final Summary

In [11]:
top_signal_horizon_rows = (
    scores.assign(abs_mean_ic=scores['mean_ic'].abs())
    .sort_values('abs_mean_ic', ascending=False)
    .head(10)
)
status_counts = scoring_gate['status'].value_counts().rename_axis('status').reset_index(name='count')

final_summary = pd.DataFrame(
    [
        {'metric': 'scoring_run_id', 'value': run_id},
        {'metric': 'scoring_version', 'value': SCORING_VERSION},
        {'metric': 'signals_scored', 'value': scores['signal_name'].nunique()},
        {'metric': 'horizons_scored', 'value': len(HORIZONS)},
        {'metric': 'score_rows', 'value': len(scores)},
        {'metric': 'sqlite_tables_written', 'value': ', '.join(sqlite_tables_written['current_table'].tolist() + sqlite_tables_written['history_table'].tolist())},
    ]
)

print('Scoring run summary')
display(final_summary)

print('Approved / watchlist / rejected counts')
display(status_counts)

print('Top 10 signal-horizon rows by abs(mean_ic)')
display(top_signal_horizon_rows)

print('Best horizon summary')
display(best_horizon_summary)

print('Family-level average abs IC')
display(family_summary)

print('SQLite tables written')
display(sqlite_tables_written)


Scoring run summary


,metric,value
0,scoring_run_id,signal_scoring_20260510_222922
1,scoring_version,phase2_signal_scoring_v2
2,signals_scored,23
3,horizons_scored,4
4,score_rows,92
5,sqlite_tables_written,"signal_scores_current, signal_score_summary_cu..."


Approved / watchlist / rejected counts


,status,count
0,REJECTED_LOW_SIGNAL,76
1,WATCHLIST,16


Top 10 signal-horizon rows by abs(mean_ic)


...

,signal_name,signal_family,best_horizon,best_mean_ic,best_abs_mean_ic,best_ic_ir,best_positive_ic_rate,best_hit_rate,signal_direction,signal_strength
0,expanded_reversal_1d,mean_reversion,1,0.017276,0.017276,0.084684,0.529553,0.505222,POSITIVE_EDGE,WEAK
1,expanded_residual_market_return_20,residual_relative_value,20,0.016748,0.016748,0.088930,0.528145,0.501801,POSITIVE_EDGE,WEAK
2,vol_of_vol_20,volatility_structure,20,0.015976,0.015976,0.128788,0.553403,0.501826,POSITIVE_EDGE,WEAK
3,expanded_reversal_3d,mean_reversion,1,0.014939,0.014939,0.070354,0.521886,0.503985,POSITIVE_EDGE,WEAK
4,range_expansion_failure_5,volatility_structure,20,0.014333,0.014333,0.112310,0.543756,0.497760,POSITIVE_EDGE,WEAK
5,expanded_distance_ma_10,mean_reversion,1,0.014215,0.014215,0.065670,0.522137,0.503944,POSITIVE_EDGE,WEAK
6,range_compression_breakout_10,microstructure_lite,20,-0.014039,0.014039,-0.096098,0.464586,0.496183,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK
7,intraday_reversal_strength_1,true_short_term_reversal,1,0.014005,0.014005,0.080810,0.530553,0.504589,POSITIVE_EDGE,WEAK
8,failed_breakout_reversal_20,microstructure_lite,20,-0.013674,0.013674,-0.097765,0.456244,0.499829,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK
9,residual_return_vs_universe_20,cross_sectional_relative_value,20,-0.013551,0.013551,-0.077233,0.481038,0.497279,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK


Family-level average abs IC


,signal_family,n_signal_horizons,avg_abs_mean_ic,max_abs_mean_ic,best_signal_name,best_horizon
0,mean_reversion,24,0.010422,0.017276,expanded_reversal_1d,1
1,residual_relative_value,8,0.008015,0.016748,expanded_residual_market_return_20,20
2,volatility_structure,12,0.007964,0.015976,vol_of_vol_20,20
3,microstructure_lite,12,0.008722,0.014039,range_compression_breakout_10,20
4,true_short_term_reversal,12,0.005701,0.014005,intraday_reversal_strength_1,1
5,cross_sectional_relative_value,12,0.008923,0.013551,residual_return_vs_universe_20,20
6,liquidity_flow,12,0.004905,0.011528,price_impact_proxy_20,20


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,scores,signal_scores_current,signal_scores_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,signal_score_summary_current,signal_score_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,gate,signal_scoring_gate_current,signal_scoring_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,best_horizon,signal_best_horizon_current,signal_best_horizon_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,family_summary,signal_scoring_family_summary_current,signal_scoring_family_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
